# Tests: `RandomMaskedViews` and `RandomMaskedViewsd`

This notebook validates the functionality of the two view-sampling transforms:

- **`RandomMaskedViews`** — extracts two random overlapping views (2D and 3D)
- **`RandomMaskedViewsd`** — dictionary-based wrapper for multiple input keys

Tests cover:
1. Output shapes for views and coordinate masks
2. Crop bounds: views always fit within the source patch
3. Per-axis overlap: the intersection satisfies the configured `overlap` fraction
4. Content correctness: extracted views match direct slices of the source patch
5. Mask correctness: mask coordinates match the true overlap region in each view frame
6. Sampling variation: view locations change across repeated calls (not fixed)
7. Seeded reproducibility: deterministic sequences under a fixed random state

## 1. Import Test Dependencies and Transform Classes

In [1]:
import sys
import os
import math
import numpy as np
import torch
import pandas as pd
from collections import Counter

# Make sure the project root is on the path when running from the tests/ subfolder
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(".")), "cryosiam"))
sys.path.insert(0, os.path.abspath(".."))

from cryosiam.transforms.array import RandomMaskedViews2
from cryosiam.transforms.dictionary import RandomMaskedViewsd2

# ── compact assertion helper ─────────────────────────────────────────────────

def check(condition: bool, msg: str) -> None:
    """Assert with a clear PASS / FAIL label."""
    status = "PASS" if condition else "FAIL"
    print(f"  [{status}] {msg}")
    assert condition, msg

print("Imports OK")

pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.


Imports OK


## 2. Create Coordinate-Encoded 2D and 3D Test Patches

Each spatial voxel is assigned a unique integer encoding its coordinates:

- **2D**: `patch[0, y, x] = y * W + x`
- **3D**: `patch[0, z, y, x] = z * H * W + y * W + x`

This lets us recover the crop start position directly from the first element of any extracted view, which is used by the helper functions in the next section.

Dictionary inputs pair an `"image"` with a `"noisy_image"` (offset by a large constant) so we can distinguish them and detect whether the dictionary wrapper uses the same crop for both keys.

In [2]:
# ── 2D patch ─────────────────────────────────────────────────────────────────
H2, W2 = 64, 64
coords_2d = np.arange(H2 * W2, dtype=np.float32).reshape(1, H2, W2)  # shape (1, H, W)
patch_2d = torch.from_numpy(coords_2d)

# dictionary version: noisy_image is the same encoding plus a large offset
DICT_OFFSET = 10_000
patch_2d_dict = {
    "image":       patch_2d.clone(),
    "noisy_image": patch_2d.clone() + DICT_OFFSET,
}

# ── 3D patch ─────────────────────────────────────────────────────────────────
D3, H3, W3 = 32, 32, 32
coords_3d = np.arange(D3 * H3 * W3, dtype=np.float32).reshape(1, D3, H3, W3)
patch_3d = torch.from_numpy(coords_3d)

patch_3d_dict = {
    "image":       patch_3d.clone(),
    "noisy_image": patch_3d.clone() + DICT_OFFSET,
}

print(f"2D patch shape : {tuple(patch_2d.shape)}  (channel, H, W)")
print(f"3D patch shape : {tuple(patch_3d.shape)}  (channel, D, H, W)")
print(f"2D dict keys   : {list(patch_2d_dict.keys())}")
print(f"3D dict keys   : {list(patch_3d_dict.keys())}")

2D patch shape : (1, 64, 64)  (channel, H, W)
3D patch shape : (1, 32, 32, 32)  (channel, D, H, W)
2D dict keys   : ['image', 'noisy_image']
3D dict keys   : ['image', 'noisy_image']


## 3. Helper Functions: Recover Start Indices and Validate Overlap

In [3]:
def infer_start_2d(view: torch.Tensor, W: int) -> tuple:
    """Recover (y0, x0) from a coordinate-encoded 2D view (shape: 1, vh, vw)."""
    first = int(view[0, 0, 0].item())
    y0 = first // W
    x0 = first % W
    return (y0, x0)


def infer_start_3d(view: torch.Tensor, H: int, W: int) -> tuple:
    """Recover (z0, y0, x0) from a coordinate-encoded 3D view (shape: 1, vd, vh, vw)."""
    first = int(view[0, 0, 0, 0].item())
    z0 = first // (H * W)
    y0 = (first % (H * W)) // W
    x0 = first % W
    return (z0, y0, x0)


def compute_overlap_lengths(start1: tuple, start2: tuple, view_size: list) -> list:
    """Return the per-axis intersection lengths for two views of equal size."""
    return [max(0, min(s1 + vs, s2 + vs) - max(s1, s2)) for s1, s2, vs in zip(start1, start2, view_size)]


def compute_overlap_count(start1: tuple, start2: tuple, view_size: list) -> int:
    """Return the exact overlap area/volume in pixels/voxels."""
    lengths = compute_overlap_lengths(start1, start2, view_size)
    return int(np.prod(lengths))


def expected_mask_from_starts(this_start: tuple, other_start: tuple, view_size: list) -> torch.Tensor:
    """Build expected inclusive [start, end] overlap coordinates in local view frame."""
    coords = []
    for s_a, s_b, vs in zip(this_start, other_start, view_size):
        a0, a1 = s_a, s_a + vs - 1
        b0, b1 = s_b, s_b + vs - 1
        ov0 = max(a0, b0)
        ov1 = min(a1, b1)
        coords.append([ov0 - a0, ov1 - a0])
    return torch.tensor(coords, dtype=torch.long)


def assert_mask_matches_overlap(mask: torch.Tensor, this_start: tuple, other_start: tuple,
                               view_size: list, label: str) -> None:
    """Check that mask encodes overlap as [spatial_dims, 2] inclusive local coordinates."""
    expected = expected_mask_from_starts(this_start, other_start, view_size)
    check(tuple(mask.shape) == tuple(expected.shape),
          f"{label}: mask shape {tuple(mask.shape)} == {tuple(expected.shape)}")
    check(mask.dtype in (torch.int64, torch.long, torch.int32),
          f"{label}: integer dtype ({mask.dtype})")
    check(torch.equal(mask.cpu().long(), expected),
          f"{label}: mask coordinates match expected overlap {expected.tolist()}")


def assert_exact_overlap(start1: tuple, start2: tuple, view_size: list, overlap: float, label: str) -> None:
    target = overlap * np.prod(view_size)
    check(np.isclose(target, round(target)), f"{label}: overlap target {target} is exactly representable")
    target = int(round(target))
    observed = compute_overlap_count(start1, start2, view_size)
    check(observed == target, f"{label}: overlap count {observed} == exact target {target}")


def assert_view_fits_in_patch(start: tuple, view_size: list, patch_size: list, label: str) -> None:
    for ax, (s, vs, ps) in enumerate(zip(start, view_size, patch_size)):
        check(s >= 0, f"{label} axis {ax}: start {s} >= 0")
        check(s + vs <= ps, f"{label} axis {ax}: end {s + vs} <= patch size {ps}")


def assert_view_matches_source(view: torch.Tensor, source: torch.Tensor,
                                start: tuple, view_size: list, label: str) -> None:
    """
    Verify that `view` equals the directly sliced region from `source`.
    Works for both 2D (source: 1,H,W) and 3D (source: 1,D,H,W).
    """
    slices = [slice(None)] + [slice(s, s + vs) for s, vs in zip(start, view_size)]
    expected = source[tuple(slices)]
    matches = torch.allclose(view.float(), expected.float())
    check(matches, f"{label}: view content matches source slice")


def assert_shapes(view1, view2, mask1, mask2, view_size: list, spatial_dims: int, label: str) -> None:
    expected_view = tuple([1] + view_size)
    expected_mask = (spatial_dims, 2)
    check(tuple(view1.shape) == expected_view, f"{label} view1 shape {tuple(view1.shape)} == {expected_view}")
    check(tuple(view2.shape) == expected_view, f"{label} view2 shape {tuple(view2.shape)} == {expected_view}")
    check(tuple(mask1.shape) == expected_mask, f"{label} mask1 shape {tuple(mask1.shape)} == {expected_mask}")
    check(tuple(mask2.shape) == expected_mask, f"{label} mask2 shape {tuple(mask2.shape)} == {expected_mask}")


print("Helpers defined.")

Helpers defined.


## 4. Test `RandomMaskedViews`: Valid Crop Bounds and Exact Overlap in 2D

Parameters: patch `(1, 64, 64)`, view size `[32, 32]`, overlap `0.5`.
Each call is independently checked for shape, bounds, content, and exact overlap area.

In [4]:
PATCH_2D     = [H2, W2]        # [64, 64]
VIEW_SIZE_2D = [32, 32]
OVERLAP_2D   = 0.5
N_SAMPLES    = 50

transform_2d = RandomMaskedViews2(
    input_image_size=PATCH_2D,
    view_size=VIEW_SIZE_2D,
    overlap=OVERLAP_2D,
)

print(f"Testing RandomMaskedViews 2D — {N_SAMPLES} samples\n")
for i in range(N_SAMPLES):
    result = transform_2d(patch_2d)

    view1, view2 = result["view1"], result["view2"]
    mask1, mask2 = result["mask1"], result["mask2"]

    # 1. Shape checks
    assert_shapes(view1, view2, mask1, mask2, VIEW_SIZE_2D, spatial_dims=2, label=f"sample {i}")

    # 2. Recover start positions from encoded content
    s1 = infer_start_2d(view1, W2)
    s2 = infer_start_2d(view2, W2)

    # 3. Bounds check
    assert_view_fits_in_patch(s1, VIEW_SIZE_2D, PATCH_2D, f"sample {i} view1")
    assert_view_fits_in_patch(s2, VIEW_SIZE_2D, PATCH_2D, f"sample {i} view2")

    # 4. Content correctness
    assert_view_matches_source(view1, patch_2d, s1, VIEW_SIZE_2D, f"sample {i} view1")
    assert_view_matches_source(view2, patch_2d, s2, VIEW_SIZE_2D, f"sample {i} view2")

    # 5. Exact overlap area check
    assert_exact_overlap(s1, s2, VIEW_SIZE_2D, OVERLAP_2D, f"sample {i}")

    # 6. Mask coordinates must encode overlap in local view frames
    assert_mask_matches_overlap(mask1, s1, s2, VIEW_SIZE_2D, f"sample {i} mask1")
    assert_mask_matches_overlap(mask2, s2, s1, VIEW_SIZE_2D, f"sample {i} mask2")

print("\nAll 2D bounds + exact-overlap + mask-coordinate checks passed.")

Testing RandomMaskedViews 2D — 50 samples

  [PASS] sample 0 view1 shape (1, 32, 32) == (1, 32, 32)
  [PASS] sample 0 view2 shape (1, 32, 32) == (1, 32, 32)
  [PASS] sample 0 mask1 shape (2, 2) == (2, 2)
  [PASS] sample 0 mask2 shape (2, 2) == (2, 2)
  [PASS] sample 0 view1 axis 0: start 9 >= 0
  [PASS] sample 0 view1 axis 0: end 41 <= patch size 64
  [PASS] sample 0 view1 axis 1: start 10 >= 0
  [PASS] sample 0 view1 axis 1: end 42 <= patch size 64
  [PASS] sample 0 view2 axis 0: start 9 >= 0
  [PASS] sample 0 view2 axis 0: end 41 <= patch size 64
  [PASS] sample 0 view2 axis 1: start 26 >= 0
  [PASS] sample 0 view2 axis 1: end 58 <= patch size 64
  [PASS] sample 0 view1: view content matches source slice
  [PASS] sample 0 view2: view content matches source slice
  [PASS] sample 0: overlap target 512.0 is exactly representable
  [PASS] sample 0: overlap count 512 == exact target 512
  [PASS] sample 0 mask1: mask shape (2, 2) == (2, 2)
  [PASS] sample 0 mask1: integer dtype (torch.int6

## 5. Test `RandomMaskedViews`: Sampling Variation and Seeded Reproducibility in 2D

**Variation** — run many calls and count distinct start positions to confirm the transform
samples more than one location.

**Reproducibility** — set the same seed on two separate transform instances and verify
their output sequences are identical.

In [5]:
N_VAR = 200

locs_v1_2d, locs_v2_2d = [], []

t2d = RandomMaskedViews2(input_image_size=PATCH_2D, view_size=VIEW_SIZE_2D, overlap=OVERLAP_2D)

for _ in range(N_VAR):
    res = t2d(patch_2d)
    locs_v1_2d.append(infer_start_2d(res["view1"], W2))
    locs_v2_2d.append(infer_start_2d(res["view2"], W2))

unique_v1 = len(set(locs_v1_2d))
unique_v2 = len(set(locs_v2_2d))

print(f"2D Sampling Variation over {N_VAR} calls:")
print(f"  View1 unique locations : {unique_v1}")
print(f"  View2 unique locations : {unique_v2}")

check(unique_v1 > 1, f"View1 samples multiple locations (got {unique_v1} unique)")
check(unique_v2 > 1, f"View2 samples multiple locations (got {unique_v2} unique)")

# ── Top-5 most frequent locations ────────────────────────────────────────────
print("\n  View1 top-5 locations:", Counter(locs_v1_2d).most_common(5))
print("  View2 top-5 locations:", Counter(locs_v2_2d).most_common(5))

2D Sampling Variation over 200 calls:
  View1 unique locations : 186
  View2 unique locations : 187
  [PASS] View1 samples multiple locations (got 186 unique)
  [PASS] View2 samples multiple locations (got 187 unique)

  View1 top-5 locations: [((29, 16), 2), ((31, 2), 2), ((17, 32), 2), ((12, 16), 2), ((11, 27), 2)]
  View2 top-5 locations: [((17, 16), 3), ((31, 18), 2), ((11, 11), 2), ((11, 3), 2), ((22, 22), 2)]


In [6]:
# ── Seeded reproducibility test (2D) ─────────────────────────────────────────
print("\n2D Seeded Reproducibility:")

SEED = 42
N_SEED = 20

t_a = RandomMaskedViews2(input_image_size=PATCH_2D, view_size=VIEW_SIZE_2D, overlap=OVERLAP_2D)
t_b = RandomMaskedViews2(input_image_size=PATCH_2D, view_size=VIEW_SIZE_2D, overlap=OVERLAP_2D)
t_a.set_random_state(seed=SEED)
t_b.set_random_state(seed=SEED)

seq_a = [infer_start_2d(t_a(patch_2d)["view1"], W2) for _ in range(N_SEED)]
seq_b = [infer_start_2d(t_b(patch_2d)["view1"], W2) for _ in range(N_SEED)]

check(seq_a == seq_b, f"Seeded sequences are identical over {N_SEED} calls")
check(len(set(seq_a)) > 1, f"Seeded sequence still varies (not all identical): {len(set(seq_a))} unique")

print(f"  Seeded sequence (first 5 view1 starts): {seq_a[:5]}")


2D Seeded Reproducibility:
  [PASS] Seeded sequences are identical over 20 calls
  [PASS] Seeded sequence still varies (not all identical): 20 unique
  Seeded sequence (first 5 view1 starts): [(28, 30), (7, 22), (18, 10), (23, 19), (23, 2)]


## 6. Test `RandomMaskedViews`: Valid Crop Bounds and Exact Overlap in 3D

Parameters: patch `(1, 32, 32, 32)`, view size `[16, 16, 16]`, overlap `0.5`.
The same bound, shape, content, and exact overlap-volume checks are applied in 3D.

In [7]:
PATCH_3D     = [D3, H3, W3]   # [32, 32, 32]
VIEW_SIZE_3D = [16, 16, 16]
OVERLAP_3D   = 0.5

transform_3d = RandomMaskedViews2(
    input_image_size=PATCH_3D,
    view_size=VIEW_SIZE_3D,
    overlap=OVERLAP_3D,
)

print(f"Testing RandomMaskedViews 3D — {N_SAMPLES} samples\n")
for i in range(N_SAMPLES):
    result = transform_3d(patch_3d)

    view1, view2 = result["view1"], result["view2"]
    mask1, mask2 = result["mask1"], result["mask2"]

    assert_shapes(view1, view2, mask1, mask2, VIEW_SIZE_3D, spatial_dims=3, label=f"3D sample {i}")

    s1 = infer_start_3d(view1, H3, W3)
    s2 = infer_start_3d(view2, H3, W3)

    assert_view_fits_in_patch(s1, VIEW_SIZE_3D, PATCH_3D, f"3D sample {i} view1")
    assert_view_fits_in_patch(s2, VIEW_SIZE_3D, PATCH_3D, f"3D sample {i} view2")

    assert_view_matches_source(view1, patch_3d, s1, VIEW_SIZE_3D, f"3D sample {i} view1")
    assert_view_matches_source(view2, patch_3d, s2, VIEW_SIZE_3D, f"3D sample {i} view2")

    assert_exact_overlap(s1, s2, VIEW_SIZE_3D, OVERLAP_3D, f"3D sample {i}")

    # Mask coordinates must encode overlap in local view frames
    assert_mask_matches_overlap(mask1, s1, s2, VIEW_SIZE_3D, f"3D sample {i} mask1")
    assert_mask_matches_overlap(mask2, s2, s1, VIEW_SIZE_3D, f"3D sample {i} mask2")

print("\nAll 3D bounds + exact-overlap + mask-coordinate checks passed.")

Testing RandomMaskedViews 3D — 50 samples

  [PASS] 3D sample 0 view1 shape (1, 16, 16, 16) == (1, 16, 16, 16)
  [PASS] 3D sample 0 view2 shape (1, 16, 16, 16) == (1, 16, 16, 16)
  [PASS] 3D sample 0 mask1 shape (3, 2) == (3, 2)
  [PASS] 3D sample 0 mask2 shape (3, 2) == (3, 2)
  [PASS] 3D sample 0 view1 axis 0: start 7 >= 0
  [PASS] 3D sample 0 view1 axis 0: end 23 <= patch size 32
  [PASS] 3D sample 0 view1 axis 1: start 6 >= 0
  [PASS] 3D sample 0 view1 axis 1: end 22 <= patch size 32
  [PASS] 3D sample 0 view1 axis 2: start 4 >= 0
  [PASS] 3D sample 0 view1 axis 2: end 20 <= patch size 32
  [PASS] 3D sample 0 view2 axis 0: start 7 >= 0
  [PASS] 3D sample 0 view2 axis 0: end 23 <= patch size 32
  [PASS] 3D sample 0 view2 axis 1: start 6 >= 0
  [PASS] 3D sample 0 view2 axis 1: end 22 <= patch size 32
  [PASS] 3D sample 0 view2 axis 2: start 12 >= 0
  [PASS] 3D sample 0 view2 axis 2: end 28 <= patch size 32
  [PASS] 3D sample 0 view1: view content matches source slice
  [PASS] 3D samp

## 7. Test `RandomMaskedViews`: Sampling Variation and Seeded Reproducibility in 3D

In [9]:
locs_v1_3d, locs_v2_3d = [], []

t3d = RandomMaskedViews2(input_image_size=PATCH_3D, view_size=VIEW_SIZE_3D, overlap=OVERLAP_3D)

for _ in range(N_VAR):
    res = t3d(patch_3d)
    locs_v1_3d.append(infer_start_3d(res["view1"], H3, W3))
    locs_v2_3d.append(infer_start_3d(res["view2"], H3, W3))

unique_v1_3d = len(set(locs_v1_3d))
unique_v2_3d = len(set(locs_v2_3d))

print(f"3D Sampling Variation over {N_VAR} calls:")
print(f"  View1 unique (z,y,x) locations : {unique_v1_3d}")
print(f"  View2 unique (z,y,x) locations : {unique_v2_3d}")

check(unique_v1_3d > 1, f"3D View1 samples multiple locations (got {unique_v1_3d} unique)")
check(unique_v2_3d > 1, f"3D View2 samples multiple locations (got {unique_v2_3d} unique)")

print("\n  View1 top-5 (z,y,x):", Counter(locs_v1_3d).most_common(5))
print("  View2 top-5 (z,y,x):", Counter(locs_v2_3d).most_common(5))

# ── Seeded reproducibility (3D) ───────────────────────────────────────────────
print("\n3D Seeded Reproducibility:")
ta3 = RandomMaskedViews2(input_image_size=PATCH_3D, view_size=VIEW_SIZE_3D, overlap=OVERLAP_3D)
tb3 = RandomMaskedViews2(input_image_size=PATCH_3D, view_size=VIEW_SIZE_3D, overlap=OVERLAP_3D)
ta3.set_random_state(seed=SEED)
tb3.set_random_state(seed=SEED)

seq_a3 = [infer_start_3d(ta3(patch_3d)["view1"], H3, W3) for _ in range(N_SEED)]
seq_b3 = [infer_start_3d(tb3(patch_3d)["view1"], H3, W3) for _ in range(N_SEED)]

check(seq_a3 == seq_b3, f"3D seeded sequences are identical over {N_SEED} calls")
check(len(set(seq_a3)) > 1, f"3D seeded sequence still varies: {len(set(seq_a3))} unique")

print(f"  3D seeded sequence (first 5): {seq_a3[:5]}")

3D Sampling Variation over 200 calls:
  View1 unique (z,y,x) locations : 194
  View2 unique (z,y,x) locations : 194
  [PASS] 3D View1 samples multiple locations (got 194 unique)
  [PASS] 3D View2 samples multiple locations (got 194 unique)

  View1 top-5 (z,y,x): [((6, 15, 4), 3), ((4, 5, 7), 2), ((15, 13, 15), 2), ((6, 10, 11), 2), ((4, 4, 16), 2)]
  View2 top-5 (z,y,x): [((6, 15, 12), 3), ((4, 5, 15), 2), ((15, 13, 7), 2), ((6, 10, 3), 2), ((4, 4, 8), 2)]

3D Seeded Reproducibility:
  [PASS] 3D seeded sequences are identical over 20 calls
  [PASS] 3D seeded sequence still varies: 20 unique
  3D seeded sequence (first 5): [(14, 10, 15), (6, 10, 15), (3, 7, 15), (1, 11, 13), (0, 11, 5)]


## 8. Test `RandomMaskedViewsd`: Output Keys, Shapes, Shared Positions, and Mask Coordinates in 2D

The dictionary wrapper should produce:  
`image_1`, `image_2`, `noisy_image_1`, `noisy_image_2`, `mask_1`, `mask_2`

`RandomMaskedViewsd` calls `self.masker.randomize()` once and then uses `_extract_view` /
`_create_mask` directly - bypassing `RandomMaskedViews.__call__()` which would re-randomize.
This guarantees that `image` and `noisy_image` always receive **identical** crop positions,
and that `mask_1` / `mask_2` encode the overlap coordinates for those positions.

In [10]:
td_2d = RandomMaskedViewsd2(
    keys=["image", "noisy_image"],
    input_image_size=PATCH_2D,
    view_size=VIEW_SIZE_2D,
    overlap=OVERLAP_2D,
)

print("Testing RandomMaskedViewsd 2D — output keys, shapes, and crop-position sharing\n")

# ── Expected output keys ──────────────────────────────────────────────────────
result_d2 = td_2d(patch_2d_dict)

expected_keys = {"image_1", "image_2", "noisy_image_1", "noisy_image_2", "mask_1", "mask_2"}
check(expected_keys == set(result_d2.keys()),
      f"Output keys match expected: {sorted(result_d2.keys())}")

# ── Shape checks ──────────────────────────────────────────────────────────────
assert_shapes(result_d2["image_1"], result_d2["image_2"],
              result_d2["mask_1"],  result_d2["mask_2"],
              VIEW_SIZE_2D, spatial_dims=2, label="dict-2D image")
assert_shapes(result_d2["noisy_image_1"], result_d2["noisy_image_2"],
              result_d2["mask_1"],         result_d2["mask_2"],
              VIEW_SIZE_2D, spatial_dims=2, label="dict-2D noisy_image")

# ── Recover crop positions from both keys ────────────────────────────────────
s_img_v1  = infer_start_2d(result_d2["image_1"],       W2)
s_img_v2  = infer_start_2d(result_d2["image_2"],       W2)
# noisy_image was offset by DICT_OFFSET; subtract before decoding
s_noisy_v1 = infer_start_2d(result_d2["noisy_image_1"] - DICT_OFFSET, W2)
s_noisy_v2 = infer_start_2d(result_d2["noisy_image_2"] - DICT_OFFSET, W2)

print(f"\n  image    view1 start : {s_img_v1},  view2 start : {s_img_v2}")
print(f"  noisy    view1 start : {s_noisy_v1},  view2 start : {s_noisy_v2}")

# ── Bounds checks ─────────────────────────────────────────────────────────────
for start, label in [(s_img_v1, "image_view1"), (s_img_v2, "image_view2"),
                     (s_noisy_v1, "noisy_view1"), (s_noisy_v2, "noisy_view2")]:
    assert_view_fits_in_patch(start, VIEW_SIZE_2D, PATCH_2D, label)

# ── Content correctness (image key) ──────────────────────────────────────────
assert_view_matches_source(result_d2["image_1"], patch_2d_dict["image"],
                            s_img_v1, VIEW_SIZE_2D, "dict-2D image_1 content")
assert_view_matches_source(result_d2["image_2"], patch_2d_dict["image"],
                            s_img_v2, VIEW_SIZE_2D, "dict-2D image_2 content")

# ── Both keys must share the same crop positions ──────────────────────────────
check(s_img_v1 == s_noisy_v1,
      f"view1 positions shared: img={s_img_v1} noisy={s_noisy_v1}")
check(s_img_v2 == s_noisy_v2,
      f"view2 positions shared: img={s_img_v2} noisy={s_noisy_v2}")

# ── Mask coordinates must match overlap region in each view frame ─────────────
assert_mask_matches_overlap(result_d2["mask_1"], s_img_v1, s_img_v2, VIEW_SIZE_2D, "dict-2D mask_1")
assert_mask_matches_overlap(result_d2["mask_2"], s_img_v2, s_img_v1, VIEW_SIZE_2D, "dict-2D mask_2")

Testing RandomMaskedViewsd 2D — output keys, shapes, and crop-position sharing

  [PASS] Output keys match expected: ['image_1', 'image_2', 'mask_1', 'mask_2', 'noisy_image_1', 'noisy_image_2']
  [PASS] dict-2D image view1 shape (1, 32, 32) == (1, 32, 32)
  [PASS] dict-2D image view2 shape (1, 32, 32) == (1, 32, 32)
  [PASS] dict-2D image mask1 shape (2, 2) == (2, 2)
  [PASS] dict-2D image mask2 shape (2, 2) == (2, 2)
  [PASS] dict-2D noisy_image view1 shape (1, 32, 32) == (1, 32, 32)
  [PASS] dict-2D noisy_image view2 shape (1, 32, 32) == (1, 32, 32)
  [PASS] dict-2D noisy_image mask1 shape (2, 2) == (2, 2)
  [PASS] dict-2D noisy_image mask2 shape (2, 2) == (2, 2)

  image    view1 start : (32, 12),  view2 start : (32, 28)
  noisy    view1 start : (32, 12),  view2 start : (32, 28)
  [PASS] image_view1 axis 0: start 32 >= 0
  [PASS] image_view1 axis 0: end 64 <= patch size 64
  [PASS] image_view1 axis 1: start 12 >= 0
  [PASS] image_view1 axis 1: end 44 <= patch size 64
  [PASS] image_

In [11]:
# ── Repeated 2D dict calls: confirm positions agree 100% across keys ──────────
n_agree_v1_2d, n_agree_v2_2d = 0, 0

for _ in range(N_VAR):
    r = td_2d(patch_2d_dict)
    sv1_img   = infer_start_2d(r["image_1"],               W2)
    sv1_noisy = infer_start_2d(r["noisy_image_1"] - DICT_OFFSET, W2)
    sv2_img   = infer_start_2d(r["image_2"],               W2)
    sv2_noisy = infer_start_2d(r["noisy_image_2"] - DICT_OFFSET, W2)
    if sv1_img == sv1_noisy:
        n_agree_v1_2d += 1
    if sv2_img == sv2_noisy:
        n_agree_v2_2d += 1

pct_v1 = 100 * n_agree_v1_2d / N_VAR
pct_v2 = 100 * n_agree_v2_2d / N_VAR
print(f"\n2D dict: over {N_VAR} calls,")
print(f"  view1 crop agreed across 'image' and 'noisy_image' : {n_agree_v1_2d}/{N_VAR} ({pct_v1:.1f}%)")
print(f"  view2 crop agreed across 'image' and 'noisy_image' : {n_agree_v2_2d}/{N_VAR} ({pct_v2:.1f}%)")

check(n_agree_v1_2d == N_VAR,
      f"2D dict view1: all {N_VAR} calls share positions ({n_agree_v1_2d}/{N_VAR})")
check(n_agree_v2_2d == N_VAR,
      f"2D dict view2: all {N_VAR} calls share positions ({n_agree_v2_2d}/{N_VAR})")


2D dict: over 200 calls,
  view1 crop agreed across 'image' and 'noisy_image' : 200/200 (100.0%)
  view2 crop agreed across 'image' and 'noisy_image' : 200/200 (100.0%)
  [PASS] 2D dict view1: all 200 calls share positions (200/200)
  [PASS] 2D dict view2: all 200 calls share positions (200/200)


## 9. Test `RandomMaskedViewsd`: Output Keys, Shapes, Shared Positions, and Mask Coordinates in 3D

In [12]:
td_3d = RandomMaskedViewsd2(
    keys=["image", "noisy_image"],
    input_image_size=PATCH_3D,
    view_size=VIEW_SIZE_3D,
    overlap=OVERLAP_3D,
)

print("Testing RandomMaskedViewsd 3D — output keys, shapes, and crop-position sharing\n")

result_d3 = td_3d(patch_3d_dict)

check(expected_keys == set(result_d3.keys()),
      f"3D dict output keys match expected: {sorted(result_d3.keys())}")

assert_shapes(result_d3["image_1"], result_d3["image_2"],
              result_d3["mask_1"],  result_d3["mask_2"],
              VIEW_SIZE_3D, spatial_dims=3, label="dict-3D image")
assert_shapes(result_d3["noisy_image_1"], result_d3["noisy_image_2"],
              result_d3["mask_1"],         result_d3["mask_2"],
              VIEW_SIZE_3D, spatial_dims=3, label="dict-3D noisy_image")

s3_img_v1  = infer_start_3d(result_d3["image_1"],                    H3, W3)
s3_img_v2  = infer_start_3d(result_d3["image_2"],                    H3, W3)
s3_noisy_v1 = infer_start_3d(result_d3["noisy_image_1"] - DICT_OFFSET, H3, W3)
s3_noisy_v2 = infer_start_3d(result_d3["noisy_image_2"] - DICT_OFFSET, H3, W3)

print(f"\n  image  view1 start : {s3_img_v1},  view2 start : {s3_img_v2}")
print(f"  noisy  view1 start : {s3_noisy_v1},  view2 start : {s3_noisy_v2}")

for start, label in [(s3_img_v1, "3D img_v1"), (s3_img_v2, "3D img_v2"),
                     (s3_noisy_v1, "3D noisy_v1"), (s3_noisy_v2, "3D noisy_v2")]:
    assert_view_fits_in_patch(start, VIEW_SIZE_3D, PATCH_3D, label)

assert_view_matches_source(result_d3["image_1"], patch_3d_dict["image"],
                            s3_img_v1, VIEW_SIZE_3D, "dict-3D image_1 content")
assert_view_matches_source(result_d3["image_2"], patch_3d_dict["image"],
                            s3_img_v2, VIEW_SIZE_3D, "dict-3D image_2 content")

check(s3_img_v1 == s3_noisy_v1,
      f"3D view1 positions shared: img={s3_img_v1} noisy={s3_noisy_v1}")
check(s3_img_v2 == s3_noisy_v2,
      f"3D view2 positions shared: img={s3_img_v2} noisy={s3_noisy_v2}")

# ── Mask coordinates must match overlap region in each view frame ─────────────
assert_mask_matches_overlap(result_d3["mask_1"], s3_img_v1, s3_img_v2, VIEW_SIZE_3D, "dict-3D mask_1")
assert_mask_matches_overlap(result_d3["mask_2"], s3_img_v2, s3_img_v1, VIEW_SIZE_3D, "dict-3D mask_2")

# ── Repeated agreement check ──────────────────────────────────────────────────
n_agree_v1_3d, n_agree_v2_3d = 0, 0
for _ in range(N_VAR):
    r = td_3d(patch_3d_dict)
    sv1_img   = infer_start_3d(r["image_1"],                    H3, W3)
    sv1_noisy = infer_start_3d(r["noisy_image_1"] - DICT_OFFSET, H3, W3)
    sv2_img   = infer_start_3d(r["image_2"],                    H3, W3)
    sv2_noisy = infer_start_3d(r["noisy_image_2"] - DICT_OFFSET, H3, W3)
    if sv1_img == sv1_noisy:
        n_agree_v1_3d += 1
    if sv2_img == sv2_noisy:
        n_agree_v2_3d += 1

pct3_v1 = 100 * n_agree_v1_3d / N_VAR
pct3_v2 = 100 * n_agree_v2_3d / N_VAR
print(f"\n3D dict: over {N_VAR} calls,")
print(f"  view1 crop agreed : {n_agree_v1_3d}/{N_VAR} ({pct3_v1:.1f}%)")
print(f"  view2 crop agreed : {n_agree_v2_3d}/{N_VAR} ({pct3_v2:.1f}%)")

check(n_agree_v1_3d == N_VAR,
      f"3D dict view1: all {N_VAR} calls share positions ({n_agree_v1_3d}/{N_VAR})")
check(n_agree_v2_3d == N_VAR,
      f"3D dict view2: all {N_VAR} calls share positions ({n_agree_v2_3d}/{N_VAR})")

Testing RandomMaskedViewsd 3D — output keys, shapes, and crop-position sharing

  [PASS] 3D dict output keys match expected: ['image_1', 'image_2', 'mask_1', 'mask_2', 'noisy_image_1', 'noisy_image_2']
  [PASS] dict-3D image view1 shape (1, 16, 16, 16) == (1, 16, 16, 16)
  [PASS] dict-3D image view2 shape (1, 16, 16, 16) == (1, 16, 16, 16)
  [PASS] dict-3D image mask1 shape (3, 2) == (3, 2)
  [PASS] dict-3D image mask2 shape (3, 2) == (3, 2)
  [PASS] dict-3D noisy_image view1 shape (1, 16, 16, 16) == (1, 16, 16, 16)
  [PASS] dict-3D noisy_image view2 shape (1, 16, 16, 16) == (1, 16, 16, 16)
  [PASS] dict-3D noisy_image mask1 shape (3, 2) == (3, 2)
  [PASS] dict-3D noisy_image mask2 shape (3, 2) == (3, 2)

  image  view1 start : (0, 5, 5),  view2 start : (0, 5, 13)
  noisy  view1 start : (0, 5, 5),  view2 start : (0, 5, 13)
  [PASS] 3D img_v1 axis 0: start 0 >= 0
  [PASS] 3D img_v1 axis 0: end 16 <= patch size 32
  [PASS] 3D img_v1 axis 1: start 5 >= 0
  [PASS] 3D img_v1 axis 1: end 21 

## 10. Stress Tests and Summary: Unique View Locations and Overall Validity

Run a large Monte Carlo loop for each class / dimensionality, track:
- Pass/fail rate for bounds and overlap checks
- Number of unique start positions visited for view1 and view2

In [16]:
N_STRESS = 1000

def stress_test(transform, patch, infer_start_fn, patch_size, view_size, overlap, label):
    """
    Run `N_STRESS` calls, checking bounds + exact overlap each time.
    Returns a dict with counts and unique-location sets.
    """
    locs_v1, locs_v2 = [], []
    n_pass_bounds = 0
    n_pass_exact_overlap = 0

    target_overlap = int(round(overlap * np.prod(view_size)))

    for _ in range(N_STRESS):
        res = transform(patch)
        v1, v2 = res["view1"], res["view2"]
        s1 = infer_start_fn(v1)
        s2 = infer_start_fn(v2)

        # bounds
        b1_ok = all(s >= 0 and s + vs <= ps
                    for s, vs, ps in zip(s1, view_size, patch_size))
        b2_ok = all(s >= 0 and s + vs <= ps
                    for s, vs, ps in zip(s2, view_size, patch_size))
        if b1_ok and b2_ok:
            n_pass_bounds += 1

        # exact overlap
        observed_overlap = compute_overlap_count(s1, s2, view_size)
        if observed_overlap == target_overlap:
            n_pass_exact_overlap += 1

        locs_v1.append(s1)
        locs_v2.append(s2)

    return {
        "label":               label,
        "total":               N_STRESS,
        "bounds_pass":         n_pass_bounds,
        "exact_overlap_pass":  n_pass_exact_overlap,
        "target_overlap":      target_overlap,
        "unique_v1":           len(set(locs_v1)),
        "unique_v2":           len(set(locs_v2)),
        "top3_v1":             Counter(locs_v1).most_common(3),
        "top3_v2":             Counter(locs_v2).most_common(3),
    }


# ── 2D stress ─────────────────────────────────────────────────────────────────
t2d_stress = RandomMaskedViews2(input_image_size=PATCH_2D, view_size=VIEW_SIZE_2D, overlap=OVERLAP_2D)
r2d = stress_test(t2d_stress, patch_2d,
                  lambda v: infer_start_2d(v, W2),
                  PATCH_2D, VIEW_SIZE_2D, OVERLAP_2D,
                  "RandomMaskedViews 2D")

# ── 3D stress ─────────────────────────────────────────────────────────────────
t3d_stress = RandomMaskedViews2(input_image_size=PATCH_3D, view_size=VIEW_SIZE_3D, overlap=OVERLAP_3D)
r3d = stress_test(t3d_stress, patch_3d,
                  lambda v: infer_start_3d(v, H3, W3),
                  PATCH_3D, VIEW_SIZE_3D, OVERLAP_3D,
                  "RandomMaskedViews 3D")

# ── dict 2D stress (image key only) ──────────────────────────────────────────
td2_stress = RandomMaskedViewsd2(keys=["image"], input_image_size=PATCH_2D,
                                 view_size=VIEW_SIZE_2D, overlap=OVERLAP_2D)

def call_dict_2d(patch):
    r = td2_stress({"image": patch})
    return {"view1": r["image_1"], "view2": r["image_2"],
            "mask1": r["mask_1"],  "mask2": r["mask_2"]}

rd_2d = stress_test(lambda p: call_dict_2d(p), patch_2d,
                    lambda v: infer_start_2d(v, W2),
                    PATCH_2D, VIEW_SIZE_2D, OVERLAP_2D,
                    "RandomMaskedViewsd 2D")

# ── dict 3D stress (image key only) ──────────────────────────────────────────
td3_stress = RandomMaskedViewsd2(keys=["image"], input_image_size=PATCH_3D,
                                 view_size=VIEW_SIZE_3D, overlap=OVERLAP_3D)

def call_dict_3d(patch):
    r = td3_stress({"image": patch})
    return {"view1": r["image_1"], "view2": r["image_2"],
            "mask1": r["mask_1"],  "mask2": r["mask_2"]}

rd_3d = stress_test(lambda p: call_dict_3d(p), patch_3d,
                    lambda v: infer_start_3d(v, H3, W3),
                    PATCH_3D, VIEW_SIZE_3D, OVERLAP_3D,
                    "RandomMaskedViewsd 3D")

print(f"Stress tests complete ({N_STRESS} samples each).  See summary below.")

Stress tests complete (1000 samples each).  See summary below.


In [17]:
# ── Summary table ─────────────────────────────────────────────────────────────
rows = []
for r in [r2d, r3d, rd_2d, rd_3d]:
    rows.append({
        "Transform":            r["label"],
        "Total samples":        r["total"],
        "Bounds pass":          f"{r['bounds_pass']}/{r['total']}",
        "Exact overlap pass":   f"{r['exact_overlap_pass']}/{r['total']}",
        "Target overlap":       r["target_overlap"],
        "Unique view1 locs":    r["unique_v1"],
        "Unique view2 locs":    r["unique_v2"],
        "Top-3 view1 starts":   str(r["top3_v1"]),
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))

print()
# Quick assertions on the aggregate results
for r in [r2d, r3d, rd_2d, rd_3d]:
    check(r["bounds_pass"] == N_STRESS,
          f"{r['label']}: all {N_STRESS} samples within bounds")
    check(r["exact_overlap_pass"] == N_STRESS,
          f"{r['label']}: all {N_STRESS} samples match exact overlap {r['target_overlap']}")
    check(r["unique_v1"] > 1,
          f"{r['label']}: view1 has multiple unique locations ({r['unique_v1']})")
    check(r["unique_v2"] > 1,
          f"{r['label']}: view2 has multiple unique locations ({r['unique_v2']})")

print("\nAll stress-test assertions passed.")

            Transform  Total samples Bounds pass Exact overlap pass  Target overlap  Unique view1 locs  Unique view2 locs                                    Top-3 view1 starts
 RandomMaskedViews 2D           1000   1000/1000          1000/1000             512                637                642          [((32, 16), 6), ((7, 16), 5), ((25, 25), 5)]
 RandomMaskedViews 3D           1000   1000/1000          1000/1000            2048                912                910    [((5, 9, 11), 3), ((4, 9, 0), 3), ((16, 7, 8), 3)]
RandomMaskedViewsd 2D           1000   1000/1000          1000/1000             512                652                651          [((18, 25), 5), ((8, 16), 5), ((22, 14), 5)]
RandomMaskedViewsd 3D           1000   1000/1000          1000/1000            2048                903                902 [((14, 15, 8), 5), ((5, 11, 5), 3), ((14, 13, 0), 3)]

  [PASS] RandomMaskedViews 2D: all 1000 samples within bounds
  [PASS] RandomMaskedViews 2D: all 1000 samples match exa